<div align="right" style="text-align: right"><i>Peter Norvig<br>April 2026</i></div>

# Project Euler Problem 3

Project Euler's [Problem #3](https://projecteuler.net/problem=3) states:


     The prime factors of 13195 are 5, 7, 13 and 29.
     What is the largest prime factor of the number 600851475143?

I'll describe here how I think about this problem, at alevel suitable for a novice programmer. There are two things to get right: the general plan of how to attack this problem, and the details of how to say it in Python.

## The General Plan

I would like to be able to find  the largest prime factor of *any* integer, not just 600851475143; I want `largest_prime_factor(n)`. Here's how I think about it:

- I don't know how to immediately find the *largest* prime factor of *n*. 
- I do know how to find the *smallest* prime factor: just go through the integers from 2 to *n*, in order. The first integer  that evenly divides *n* must be the smallest prime factor. It is a factor because it evenly divides, it is smallest because we didn't find a smaller one first, and it is prime because if it were a composite number (like 4 or 9), then we would have found one of its factors (like 2 or 3) first.
- Once I have the smallest prime factor ***p*** then I know: **the largest prime factor of *n* is the maximum of *p* and the largest prime factor of *n*/*p*.**
- If *n* is prime, then the largest prime factor is *n*.
- If *n* is 1, then [by convention](https://oeis.org/wiki/Greatest_prime_factor_of_n) we say the largest prime factor is 1, even though 1 is usually not considered a prime.

## How to Say It in Python

There are a few things to remember (or look up):
- `def` is the keyword for [defining a function](https://k21academy.com/ai-ml/functions/) in Python.
  - `def` is followed by the function name, the function parameter(s), and the function body.
  - It is good practice to include a [documentation string](https://www.educative.io/answers/what-is-a-python-docstring) as the first thing in the function body, for clarification. 
- `range(2, n)` means all the integers starting at 2 and stopping just before *n*. E.g., `range(2, 6)` is [2, 3, 4, 5].
- `n % p` means "the remainder of *n* divided by *p*"; e.g., `13 % 10` is `3`.
- `n % p == 0` means "is the remainder of *n* divided by *p* equal to zero?" or "does *p* evenly divide *n*?" or "is *p* a factor of *n*"?
- `n // p` means "integer division"; `30 // 2` is the integer 15, while `30 / 2` is the decimal number 15.0. We want integers.
- `return` means to immediately return a value from a function; don't do any more statements in the function.


## The Implementation and the Solution

The **implementation**:

In [1]:
def largest_prime_factor(n):
    """The largest prime that evenly divides n.
    Find the smallest prime p that evenly divides n, 
    and return the maximum of p and the largest prime factor of n/p.
    If n is prime or if n = 1, this will return n."""
    for p in range(2, n):
        if n % p == 0:  # n is composite
            return max(p, largest_prime_factor(n // p))
    return n            # n is prime or 1



The **solution**:

In [2]:
largest_prime_factor(600851475143)

6857

We can represent the execution of the program as a series of equations, one per line.  Using `F` as an abbreviation for `largest_prime_factor`, here is hjow we find the largest prime factor of 360:

      F(360) = max(2, F(180))
                      F(180) = max(2, F(90))
                                      F(90) = max(2, F(45))
                                                     F(45) = max(3, F(15))
                                                                    F(15) = max(3, F(5))
                                                                                   F(5) = 5
      F(36) = max(2,           max(2,          max(2,            3,             3,        5))) = 5

In [3]:
2 * 2 * 2 * 3 * 3 * 5

360

## Some Tests

Are we sure our function is correct? It is good idea to define some **tests** to gain confidence. 

In [4]:
def tests():
    """Test largest_prime_factor."""
    cases = {1: 1,         # by convention, 1
             2: 2,         # even prime
             3: 3,         # odd prime
             6: 3,         # composite
             32: 2,        # power of 2
             49: 7,        # square of a prime
             97: 97,       # bigger prime
             99991: 99991, # even bigger prime
             97**9: 97,    # even bigger power of a prime
             360: 5,       # test case for equations above
             600851475143: 6857 # Project Euler #3
            }
    correct = sum(largest_prime_factor(n) == cases[n] for n in cases)
    return [correct, 'out of', len(cases), 'tests pass']

tests()

[11, 'out of', 11, 'tests pass']

## Efficiency

How long does it take to get our answer? We can use `%time` to see it is just a few hundred microseconds (μs):

In [5]:
%time largest_prime_factor(600851475143)

CPU times: user 181 μs, sys: 0 ns, total: 181 μs
Wall time: 183 μs


6857

The algorithm is slowest when *n* is prime, because the `for` loop has to go all the way up to *n*. How long would it take for the largest 8-digit prime, 99,999,989?

In [6]:
n8 = 99999989 # An 8-digit number that happens to be prime 
%time largest_prime_factor(n8)

CPU times: user 1.9 s, sys: 18.4 ms, total: 1.92 s
Wall time: 1.92 s


99999989

About 2 seconds. Maybe that's good enough. But could we speed things up?

In trying to find a *p* that evenly divides *n*, the algorithm tests all the integers from 2 to *n*. But do we really have to test all those potential factors? No! Either *n* is prime, or it has two factors *p* and *q* such that *p* × *q* = *n*. Of those two factors, one must be less than or equal to the square root of *n*. So to determine if *n* has a prime factor other than itself, we only have to check integers up to √*n*, not all the way up to *n*. That's a big difference! for an 8-digit prime it is the difference between roughly 100 million steps versus a mere 10 thousand steps.

Let's change the definition of `largest_prime_factor` to incorporate this new trick. (We will `import` the square root function, `sqrt`, from the `math` module.)

In [7]:
from math import sqrt

def largest_prime_factor(n):
    """The largest prime that evenly divides n.
    Find the smallest prime p that evenly divides n, 
    and return the maximum of p and the largest prime factor of n/p.
    If n is prime or if n = 1, this will return n."""
    for p in range(2, int(sqrt(n) + 1)): # <<<< only need to go up to √n
        if n % p == 0:  # n is composite
            return max(p, largest_prime_factor(n // p))
    return n            # n is prime or 1

Any time you change a function, you should re-run the tests to give you some confidence that you didn't introduce a bug:

In [8]:
tests()

[11, 'out of', 11, 'tests pass']

Now we can see how fast the new function is on the 8-digit prime:

In [9]:
%time largest_prime_factor(n8)

CPU times: user 196 μs, sys: 0 ns, total: 196 μs
Wall time: 196 μs


99999989

As predicted, this is about 10,000 times faster.

We should be able to handle a 16-digit prime in about 2 seconds:

In [10]:
n16 = 9927935178558959 # Largest 16-digit prime
%time largest_prime_factor(n16)

CPU times: user 1.96 s, sys: 19.3 ms, total: 1.97 s
Wall time: 1.97 s


9927935178558959

## Imperative versus Declarative (or Functional) Style

Our definition of largest_prime_factor mixed paradigms, using some [imperative](https://en.wikipedia.org/wiki/Imperative_programming) features (a for loop with a return in the middle) and some [functional](https://en.wikipedia.org/wiki/Functional_programming) features (an implementation of the equation `largest_prime_factor(n) = max(p, largest_prime_factor(n // p))`).

Here's what it might look like if we leaned into the functional style more, making the equation more explicit:

In [11]:
def largest_prime_factor(n):
    """The largest prime that evenly divides n.
    Find the smallest prime p that evenly divides n, 
    and return the maximum of p and the largest prime factor of n/p.
    If n is prime or if n = 1, this will return n."""
    p = smallest_prime_factor(n)
    return 1 if n == 1 else max(p, largest_prime_factor(n // p))

def smallest_prime_factor(n):
    """The smallest prime that evenly divides n (or n itself if no prime divisors)."""
    return next((p for p in range(2, int(sqrt(n) + 1)) if n % p == 0), n)

tests()

[11, 'out of', 11, 'tests pass']

And here's what it looks like in an imperative style, updating `n`, `p`,  and `largest` within a `while` loop that contains  another `while` loop to take care of the case where *n* has *p* as a repeated factor.

In [12]:
def largest_prime_factor(n):
    """The largest prime that evenly divides n.
    Find the smallest prime p that evenly divides n, 
    and return the maximum of p and the largest prime factor of n/p.
    If n is prime or if n = 1, this will return n."""
    largest = 1
    p = 2
    while p * p <= n:
        while n % p == 0:
            largest = p
            n = n // p
        p = p + 1
    return max(n, largest)
    
tests()

[11, 'out of', 11, 'tests pass']